# Initialization for divisive inhibition

We want to figure out an initialization scheme for Dale's ANNs (DANNs) with divisive inhibition, such that, at initialization the mean and variance of the pre-nonlinearity activations of the output is 0 and 1, respectively.

Let us define our input->E weight matrix as $W^{EX}$ and the subtractive inhibition input->I weight matrix as $W^{IX}$ and I->E weight matrix as $W^{EI}$.
We will further define our divisive inhibition input->I weight matrix as $B^{IX}$ and I->E weight matrix as $B^{EI}$



## Subtractive-only DANN

For subtractive only DANN, we want the pre-nonlinearity activation of a layer, $Z$ as:
$$
Z = Z^E - Z^I = W^{EX} X - W^{EI} W^{IX} X = (W^{EX} - W^{EI} W^{IX}) X
$$


## [RECAP] Subtractive inhibition initialization


We initialized the matrices such that the mean of Z is 0.
So, we set the following:

$$
W^{IX} = \mathbf{1} \overline{W^{EX}}\\
W^{EI} = \frac{1}{n_i}\mathbf{1}
$$

where $\overline{W^{EX}}$ is the mean row of $W^{EX}$.


## DANN with Divisive-inhibition

For DANN with divisive inhibition, we want the pre-nonlinearity activation of a layer, $Z$ as:
$$
Z = (Z^E - Z^I) / Z^d \\
Z^E = W^{EX} X \\
Z^I = W^{EI} W^{IX} X \\
Z^d = g(B^{EI} f(B^{IX} X))
$$
where $f(.)$ and $g(.)$ are point-wise nonlinear functions, and $A/B$ indicates a point-wise division operation between vectors $A$ and $B$.

## Initializing the $B$ matrices

### Theoretically deriving the variance

To keep things simple, as above, we set $B^{EI}$ to be $\mathbf{1}$ or some constant factor of it. In doing so, we setup the problem that we have the same value of $Z^d$ in all dimensions. Therefore, we can keep the subtractive-inhibition init as is and have the mean of $Z$ to be 0.

In order to initialize $B^{IX}$ and the non-linearities, let us first compute the variance of $Z_{EI\_sub} = Z^E - Z^I$. For simplicity, let us assume that the effective weight matrix, $(W^{EX} - W^{EI} W^{IX})$ is denoted as $W$.

$$
var(Z_{EI\_sub}) = var(WX) = (WX)^T (WX) = X^T W^T W X
$$

Let us assume that $W$ has a svd as:
$$
W = U S V^T
$$
and
$$
X = V R \\
where, \quad R_i = V_i^T X
$$
Then, $Z_{EI\_sub} = WX = U S V^T V R = U S R$
Therefore,
$$
var(Z_{EI\_sub}) = \frac{1}{n_e}(USR)^T (USR) = \frac{1}{n_e} R^T S^T U^T U S R = \frac{1}{n_e} (SR)^T(SR)
$$

Let us verify this below.

P.S.: I am overloading notations here a bit. Initially I was referring to $S$ as the singular value matrix, written in a diagonal form. But now, I have switched to referring to $S$ as the singular values as a vector (as numpy does).

### Verifying variance of Z

In [ ]:
import numpy as np

In [ ]:
n_e = 5 # num excitatory
n_i = 1 # num subtractive inhib
n_input = 5

In [ ]:
# X = np.random.rand(n_e,1)
X_init = np.random.rand(n_input, 1)
X = X_init #np.random.binomial(1, 0.5, (n_input, 1))

In [ ]:
target_std_wex = np.sqrt(2*n_e/(n_input*(n_e-1)))
exp_scale = target_std_wex
W_EX = np.random.exponential(scale=exp_scale, size=(n_e, n_input))

In [ ]:
W_EX

array([[0.0494167 , 0.38662174, 1.66885811, 1.59127634, 0.71017034],
       [0.34412888, 0.16876526, 0.20756598, 0.19185803, 0.22921521],
       [0.78346152, 0.95165567, 0.53282967, 0.33463051, 0.42913788],
       [0.68360822, 0.12803091, 2.27056977, 0.01006515, 0.36832912],
       [1.41659122, 0.26306077, 0.52705889, 0.19898092, 1.53540806]])

In [ ]:
# W_EX = np.random.rand(n_e, n_e) # Check with log normal
# W_IX = np.ones((n_i, 1)) @ np.mean(W_EX, axis=0, keepdims=True)
W_IX = np.ones((n_i, 1)) @ np.mean(W_EX, axis=0, keepdims=True)
W_EI = np.ones((n_e, n_i))/n_i

In [ ]:
print(W_EX.shape, W_IX.shape, W_EI.shape)

(5, 5) (1, 5) (5, 1)


In [ ]:
W = W_EX - W_EI @ W_IX

In [ ]:
print(W.shape)

(5, 5)


In [ ]:
np.allclose(np.sum(W,axis=0), np.zeros((W.shape[1])))  # verify mean row of W is 0

True

In [ ]:
Z_E = W_EX @ X
Z_I = W_EI @ W_IX @ X
Z_EI_sub = Z_E - Z_I
print(Z_EI_sub.shape, np.mean(Z_EI_sub)) # verify mean is 0

(5, 1) 6.661338147750939e-17


In [ ]:
U, S, V_T = np.linalg.svd(W)
V = V_T.T
print(U.shape, S.shape, V.shape)
print(f"W reconstruction assert: {np.allclose(W, U @ np.diag(S) @ V.T)}")

(5, 5) (5,) (5, 5)
W reconstruction assert: True


In [ ]:
R = V.T @ X_init

In [ ]:
R.shape

(5, 1)

In [ ]:
np.diag(S)

array([[1.91981031e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00, 1.36091776e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.18729222e+00, 0.00000000e+00,
        0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 6.44522581e-01,
        0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        7.66217855e-17]])

In [ ]:
Z_EI_var_empirical = np.std(Z_EI_sub)**2

SR_product = np.diag(S) @ R
Z_EI_var_theory = SR_product.T @ SR_product/n_e
Z_EI_var_theory = Z_EI_var_theory[0,0]

print(f"Empirical variance  : {Z_EI_var_empirical}")
print(f"Theoretical variance: {Z_EI_var_theory}")
assert np.allclose(Z_EI_var_empirical, Z_EI_var_theory), "theory and empirical don't match!!"

Empirical variance  : 0.25710142700594385
Theoretical variance: 0.2571014270059438


### Initialization scheme

Great! Now that we have verified that our theoretical variance matches the empirical one, we can proceed to propose our initialization scheme.

We want an initialization scheme that works for "all" inputs, i.e. for all inputs the mean of $Z$ is 0 and variance is 1. The condition for zero-mean is already satisfied because of the init scheme of subtractive-only DANN. For unit-variance, we effectively want to compute $SR$ in the divisive inhib units and then divide each element of $Z$ by the sum of activations of the divisive inhib units.

To that end, we will set $B^{IX} = S V^T$, and $f(.)$ to be the square function.
Therefore,
$$
f(B^{IX}X) = f(S V^T V R) = f(SR) = (SR) ⊙ (SR)
$$
If we set $B^{EI}$ to be $\frac{1}{n_e} \mathbf{1}$ and $g(.)$ to be the (non-negative) square-root function,
$$
Z^d = g(B^{EI} f(B^{IX} X)) = g\left(\frac{1}{n_e} \mathbf{1} (SR) ⊙ (SR)\right) = \sqrt{\frac{1}{n_e} (SR)^T(SR)} = \sqrt{var(Z_{EI\_sub})}
$$
Therefore, dividing $Z_{EI\_sub}$ by $Z^d$ will yield a unit-variance $Z$.

Let's verify this?

### Verifying initialization scheme

In [ ]:
B_IX = np.diag(S) @ V.T
B_EI = np.ones((n_e,n_e))/n_e
Z_d_squared = B_EI @ (B_IX @ X)**2
Z_d = np.sqrt(Z_d_squared)

In [ ]:
B_IX.shape, B_EI.shape

((5, 5), (5, 5))

In [ ]:
Z = Z_EI_sub/Z_d
print(f"variance of Z_EI_sub: {np.std(Z_EI_sub)}")
print(f"variance of Z: {np.std(Z)}")
assert np.allclose(np.std(Z), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub: 0.5070517005256405
variance of Z: 1.0000000000000002


## Note on number of divisive inhibitory units

If you closely notice, the dimensionality of $B^{IX}$ and $B^{EI}$ are $(n_e, n_e)$. This implies that we need $n_e$ number of divisive inhibitory units, which is not very bio-plausible.

In [ ]:
print(B_IX.shape, B_EI.shape)

(5, 5) (5, 5)


One thing to keep in mind is that $W$ is not a full-rank matrix, as its rows sum to 0. Therefore, we can safely assume that not all values in $S$ are non-zero. We can very likely get rid of the last eigenvalue and still maintain unit-variance.

Let's try this!

In [ ]:
n_i_div = n_e - 1

B_IX = np.diag(S[:n_i_div]) @ V[:,:n_i_div].T
B_EI = np.ones((n_e,n_i_div))/n_e
Z_d_squared = B_EI @ (B_IX @ X)**2
Z_d = np.sqrt(Z_d_squared)

In [ ]:
print(B_IX.shape, B_EI.shape)

(4, 5) (5, 4)


In [ ]:
Z = Z_EI_sub/Z_d
print(f"variance of Z_EI_sub: {np.std(Z_EI_sub)}")
print(f"variance of Z: {np.std(Z)}")
assert np.allclose(np.std(Z), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub: 0.5070517005256405
variance of Z: 1.0000000000000002


Going even lower will start to hurt the unit-variance condition.

In [ ]:
n_i_div = n_e - 2

B_IX = np.diag(S[:n_i_div]) @ V[:,:n_i_div].T
B_EI = np.ones((n_e,n_i_div))/n_e
Z_d_squared = B_EI @ (B_IX @ X)**2
Z_d = np.sqrt(Z_d_squared)

In [ ]:
Z = Z_EI_sub/Z_d
print(f"variance of Z_EI_sub: {np.std(Z_EI_sub)}")
print(f"variance of Z: {np.std(Z)}")
assert np.allclose(np.std(Z), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub: 0.5070517005256405
variance of Z: 1.2448580738951431


AssertionError: Init scheme does not yield unit variance!!

## Note on balance

The previous section illustrated that it is possible to have 0-mean and unit-variance initialization with $(n_e-1)$ number of divisive inhibitory units. Lower number of inhibitory units means that the unit-variance condition may no longer be true for all inputs at initialization.

#### Can we have lesser inhibitory units?
Yes, we can but at the cost of not having a unit-variance condition for some inputs. Therefore, we now have certain "directions" in the input space for which we are no longer balanced.

Let us re-try the initialization scheme above that didn't work, i.e. with $(n_e-2)$ divisive inhibitory units. This time, we will try with two different inputs: $X_1$ and $X_2$ that are slight variants of original input $X$.

Remember that $X = V R$. We will define
$$
X_1 = V R_1 \\
X_2 = V R_2
$$
where $R_1$ is $R$ but the second dimension is set to 0 and $R_2$ is $R$ but the first dimension is set to 0.


In [ ]:
R1 = np.ones_like(R)*R
R1[1,0] = 0.0
R2 = np.ones_like(R)*R
R2[0,0] = 0.0
print("R:", R)
print("R1:", R1)
print("R2:", R2)

R: [[ 0.08150443]
 [-0.78290523]
 [ 0.73261568]]
R1: [[0.08150443]
 [0.        ]
 [0.73261568]]
R2: [[ 0.        ]
 [-0.78290523]
 [ 0.73261568]]


In [ ]:
X1 = V @ R1
X2 = V @ R2

#### Using the init scheme for fewer inhib units

In [ ]:
n_i_div = n_e - 2
B_IX = np.diag(S[:n_i_div]) @ V[:,:n_i_div].T
B_EI = np.ones((n_e,n_i_div))/n_e

#### Checking for X1

In [ ]:
Z_E1 = W_EX @ X1
Z_I1 = W_EI @ W_IX @ X1
Z_EI_sub1 = Z_E1 - Z_I1

Z_d_squared1 = B_EI @ (B_IX @ X1)**2
Z_d1 = np.sqrt(Z_d_squared1)
Z1 = Z_EI_sub1/Z_d1
print(f"variance of Z_EI_sub1: {np.std(Z_EI_sub1)}")
print(f"variance of Z1: {np.std(Z1)}")
assert np.allclose(np.std(Z1), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub1: 0.03243651164194078
variance of Z1: 1.0000000000000004


#### Checking for X2

In [ ]:
Z_E2 = W_EX @ X2
Z_I2 = W_EI @ W_IX @ X2
Z_EI_sub2 = Z_E2 - Z_I2

Z_d_squared2 = B_EI @ (B_IX @ X2)**2
Z_d2 = np.sqrt(Z_d_squared2)
Z2 = Z_EI_sub2/Z_d2
print(f"variance of Z_EI_sub2: {np.std(Z_EI_sub2)}")
print(f"variance of Z2: {np.std(Z2)}")
assert np.allclose(np.std(Z2), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub2: 0.2671059096892126
variance of Z2: 8334199459425215.0


AssertionError: Init scheme does not yield unit variance!!

#### TL;DR
The init scheme worked for $X_1$, but not $X_2$. Therefore, inputs along the direction of $X_1$ will be balanced (0-mean, unit-variance) but those along $X_2$ will not be balanced!

It is unclear right now what advantages can be obtained by being in balance in some directions vs not, but perhaps this is something gradient descent will figure out and let us know. ;)

## Note on different number of inputs and excitatory cells

So far, we have been assuming that the number of pyramidal cells is constant in each layer, i.e. $n_{input} = n_e$. But realistically, representation dimensionality can increase or decrease across layers, i.e. $n_{input} \neq n_e$. So, here we verify the initialization for rectangular $W^{EX}$ matrix.

### Case 1: $n_e$ < $n_{input}$

In [ ]:
n_e = 3 # num excitatory
n_i = 1 # num subtractive inhib
n_input = 5
n_k = min(n_input, n_e)
X_init = np.random.rand(n_input, 1)
X = X_init #np.random.binomial(1, 0.5, (n_input, 1))

In [ ]:
target_std_wex = np.sqrt(2*n_e/(n_input*(n_e-1)))
exp_scale = target_std_wex
W_EX = np.random.exponential(scale=exp_scale, size=(n_e, n_input))

In [ ]:
W_IX = np.ones((n_i, 1)) @ np.mean(W_EX, axis=0, keepdims=True)
W_EI = np.ones((n_e, n_i))/n_i
print(W_EX.shape, W_IX.shape, W_EI.shape)

(3, 5) (1, 5) (3, 1)


In [ ]:
W = W_EX - W_EI @ W_IX
print(W.shape)

(3, 5)


In [ ]:
Z_E = W_EX @ X
Z_I = W_EI @ W_IX @ X
Z_EI_sub = Z_E - Z_I
print(Z_EI_sub.shape, np.mean(Z_EI_sub)) # verify mean is 0

(3, 1) -2.220446049250313e-16


In [ ]:
U, S, V_T = np.linalg.svd(W)
U = U[:,:n_k]
V = V_T[:n_k,:].T
print(U.shape, S.shape, V.shape)
print(f"W reconstruction assert: {np.allclose(W, U @ np.diag(S) @ V.T)}")

(3, 3) (3,) (5, 3)
W reconstruction assert: True


In [ ]:
B_IX = np.diag(S) @ V.T
B_EI = np.ones((n_e,n_k))/n_e
print(B_IX.shape, B_EI.shape)
Z_d_squared = B_EI @ (B_IX @ X)**2
Z_d = np.sqrt(Z_d_squared)

(3, 5) (3, 3)


In [ ]:
Z = Z_EI_sub/Z_d
print(f"variance of Z_EI_sub: {np.std(Z_EI_sub)}")
print(f"variance of Z: {np.std(Z)}")
assert np.allclose(np.std(Z), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub: 0.7612031975104245
variance of Z: 1.0


### Case 2: $n_e$ > $n_{input}$

In [ ]:
n_e = 7 # num excitatory
n_i = 1 # num subtractive inhib
n_input = 5
n_k = min(n_input, n_e)
X_init = np.random.rand(n_input, 1)
X = X_init #np.random.binomial(1, 0.5, (n_input, 1))

In [ ]:
target_std_wex = np.sqrt(2*n_e/(n_input*(n_e-1)))
exp_scale = target_std_wex
W_EX = np.random.exponential(scale=exp_scale, size=(n_e, n_input))

In [ ]:
W_IX = np.ones((n_i, 1)) @ np.mean(W_EX, axis=0, keepdims=True)
W_EI = np.ones((n_e, n_i))/n_i
print(W_EX.shape, W_IX.shape, W_EI.shape)

(7, 5) (1, 5) (7, 1)


In [ ]:
W = W_EX - W_EI @ W_IX
print(W.shape)

(7, 5)


In [ ]:
Z_E = W_EX @ X
Z_I = W_EI @ W_IX @ X
Z_EI_sub = Z_E - Z_I
print(Z_EI_sub.shape, np.mean(Z_EI_sub)) # verify mean is 0

(7, 1) 1.7446361815538174e-16


In [ ]:
U, S, V_T = np.linalg.svd(W)
U = U[:,:n_k]
V = V_T[:n_k,:].T
print(U.shape, S.shape, V.shape)
print(f"W reconstruction assert: {np.allclose(W, U @ np.diag(S) @ V.T)}")

(7, 5) (5,) (5, 5)
W reconstruction assert: True


In [ ]:
B_IX = np.diag(S) @ V.T
B_EI = np.ones((n_e,n_k))/n_e
print(B_IX.shape, B_EI.shape)
Z_d_squared = B_EI @ (B_IX @ X)**2
Z_d = np.sqrt(Z_d_squared)

(5, 5) (7, 5)


In [ ]:
Z = Z_EI_sub/Z_d
print(f"variance of Z_EI_sub: {np.std(Z_EI_sub)}")
print(f"variance of Z: {np.std(Z)}")
assert np.allclose(np.std(Z), 1.0), "Init scheme does not yield unit variance!!"

variance of Z_EI_sub: 0.6064328055442139
variance of Z: 0.9999999999999999


## Note on negative values in $B$ matrices

We initialized $B^{EI}$ to be a multiple of $\mathbf{1}$, so no negative elements in that matrix. But $B^{IX}$ is initialized to be $S V^T$ and could have negative elements.

In [ ]:
n_i_div = n_e - 1

B_IX = np.diag(S[:n_i_div]) @ V[:,:n_i_div].T
B_EI = np.ones((n_e,n_i_div))/n_e
print(B_IX)

[[-0.46278237 -0.30006968  0.41344495]
 [-0.13674201 -0.38157468 -0.42999868]]


One potential way is to represent $B^{IX}$ as difference of two matrices, just as the subtractive-only DANN does (think of $B^{IX}$ being the effective weight matrix $W$).
Let us assume:
$$
B^{IX} = B^{IX}_1 - B^{IX}_2
$$
where all elements in both matrices are non-negative.

Note that $B^{IX}_1$ can be thought of as the feedforward inhibition pathway (Pyr -> SST) and $B^{IX}_2$ can be thought of as the disinhibition pathway (Pyr -> VIP -> SST). Using this (more complicated) architecture, we have a way of keeping all elements to be non-negative and have the subtractive/divisive inhibition incorporated in the architecture.

Unfortunately, I have not thought about the connections to cortical microcircuit beyond this. But Blake+Dan+Jonny are probably better people to discuss this with. Happy to chat more. :)

## Initialization for Divisive Recurrent Units with External Input and Bias

When the layer receives a recurrent state $h_{t-1}$, an external input $x_t$, and a bias $b$, the pre-activation is:

$$Z_t = \frac{W_{EX}^{rec}h_{t-1} + W_{EX}^{ff} x_t + b}{Z^d_t}$$

To initialize $Z^d_t$ such that $Z_t$ has unit variance, we include the bias in our weight concatenation and perform SVD on $W_{total} = [W_{eff}^{rec} \mid W_{eff}^{ff} \mid b]$.

In [ ]:
import numpy as np

n_h = 10     # hidden size
n_in = 5     # external input size

# 1. Initialize Raw Weights
W_EX_rec = np.random.randn(n_h, n_h) * (1.0 / np.sqrt(n_h))
W_EX_ff = np.random.randn(n_h, n_in) * (1.0 / np.sqrt(n_in))
bias_raw = np.random.randn(n_h, 1)

# 2. Apply Subtractive Inhibition to the whole system (including bias)
# We want the mean of (W_rec*h + W_ff*x + bias) to be 0.
# We treat the bias as a weight for an input that is always 1.
W_full_raw = np.concatenate([W_EX_rec, W_EX_ff, bias_raw], axis=1)

# Calculate subtractive inhibition for the combined weights
W_IX_full = np.ones((1, 1)) @ np.mean(W_full_raw, axis=0, keepdims=True)
W_EI_full = np.ones((n_h, 1))

W_total_eff = W_full_raw - (W_EI_full @ W_IX_full)

# 3. SVD for Divisive Weights based on the effective centered weights
U, S, VT = np.linalg.svd(W_total_eff)
B_IX_total = np.diag(S[:n_h]) @ VT[:n_h, :]
B_EI = np.ones((n_h, n_h)) / n_h

# 4. Verification
h_prev = np.random.randn(n_h, 1)
x_t = np.random.randn(n_in, 1)
x_concat = np.vstack([h_prev, x_t, [[1.0]]])

# Numerator (Centered)
z_num = W_total_eff @ x_concat

# Denominator
z_d = np.sqrt(B_EI @ (B_IX_total @ x_concat)**2)

z_final = z_num / z_d

print(f"Centered Mean: {np.mean(z_final):.4f}")
print(f"Centered Std:  {np.std(z_final):.4f}")

Centered Mean: 0.0000
Centered Std:  1.0000


### Verification against Layer Normalization

We now verify that our divisive normalization $Z_{final} = Z_{num} / Z^d$ is numerically equivalent to applying a standard Layer Norm (centering and scaling by standard deviation) to the raw linear drive.

In [ ]:
def layer_norm_simple(x):
    # Standard Layer Norm: (x - mean) / std
    return (x - np.mean(x)) / np.std(x)

# 1. Compute the raw linear drive (numerator before subtractive centering)
z_raw = W_full_raw @ x_concat

# 2. Apply standard Layer Norm
z_layernorm = layer_norm_simple(z_raw)

# 3. Compare with our DANN output (z_final from the previous cell)
diff = np.abs(z_final - z_layernorm)

print(f"Max difference between DANN and LayerNorm: {np.max(diff):.2e}")
print(f"Equivalence verified: {np.allclose(z_final, z_layernorm)}")

Max difference between DANN and LayerNorm: 1.13e-14
Equivalence verified: True


### Explicit Mean Verification

To address the concern about centering, let's verify that the subtractive inhibition step in our DANN (using `W_total_eff`) produced a zero-mean output, matching the centering behavior of Layer Norm.

In [ ]:
print(f"Mean of DANN output (z_final): {np.mean(z_final):.4e}")
print(f"Mean of LayerNorm output (z_layernorm): {np.mean(z_layernorm):.4e}")

# The centering happens here via subtractive inhibition:
# z_num = (W_raw - Mean_W) @ x
# which is equivalent to:
# z_num = (W_raw @ x) - Mean(W_raw @ x)

expected_zero = np.mean(z_final)
assert np.isclose(expected_zero, 0.0, atol=1e-10), "z_final is not zero-centered!"

Mean of DANN output (z_final): 0.0000e+00
Mean of LayerNorm output (z_layernorm): -4.4409e-17
